# Database & Pipeline Basics: Search, Load, Browse

How to populate/search the DataJoint database for experiments, pick one, and
initialize an `MEAPipeline`/`AnalysisChunk` for it, plus a look at the underlying
objects it returns (`stim`, `resp`, `analysis_chunk`) and what's in them.

Every stimulus-specific demo (contrast grating, flash, DS/OS grating, correlated
spiking) repeats this same "choose an experiment + initialize pipeline" setup on
its own, so each notebook runs standalone.


Import `retinanalysis` and the standard scientific-Python packages this notebook uses.

In [1]:
import retinanalysis as ra
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

C:\Users\Yasmine Tani\anaconda3\envs\retinanalysis\Lib\site-packages\datajoint\settings.py:992: UserWarning: No datajoint.json found. Using defaults and environment variables. Run `dj.config.save_template()` to create a template configuration.
  config = _create_config()


Optional diagnostic: uncomment to debug a raw experiment file that fails to ingest into the database.

In [2]:
# %run _diagnose_meta_json.py

Populate the DataJoint database from raw experiment files; only ingests experiments not already in the database unless `refresh_existing=True`.

In [3]:
#ra.reload_experiment_data("20260722A") 

In [ ]:
ra.populate_database(refresh_existing=True)

[2026-08-13 15:42:02] DataJoint 2.2.2 connected to root@127.0.0.1:3306


Experiments:   0%|          | 0/77 [00:00<?, ?it/s]

[2026-08-13 15:42:06] Deleting 876 rows from `schema`.`stimulus`
[2026-08-13 15:42:06] Deleting 876 rows from `schema`.`response`
[2026-08-13 15:42:09] Deleting 438 rows from `schema`.`epoch`
[2026-08-13 15:42:13] Deleting 11 rows from `schema`.`epoch_block`
[2026-08-13 15:42:13] Deleting 6 rows from `schema`.`epoch_group`
[2026-08-13 15:42:13] Deleting 0 rows from `schema`.`sorted_cell_type`
[2026-08-13 15:42:13] Deleting 1 rows from `schema`.`cell`
[2026-08-13 15:42:13] Deleting 7417 rows from `schema`.`sorted_cell`
[2026-08-13 15:42:13] Deleting 0 rows from `schema`.`cell_type_file`
[2026-08-13 15:42:13] Deleting 1 rows from `schema`.`preparation`
[2026-08-13 15:42:13] Deleting 0 rows from `schema`.`tags`
[2026-08-13 15:42:13] Deleting 11 rows from `schema`.`sorting_chunk`
[2026-08-13 15:42:13] Deleting 1 rows from `schema`.`animal`
[2026-08-13 15:42:13] Deleting 1 rows from `schema`.`experiment`
[2026-08-13 15:42:28] Deleting 81 rows from `schema`.`stimulus`
[2026-08-13 15:42:28] D

## Search for datasets

Searches the database for every SpatialNoise-protocol dataset, then filters to experiments that actually have local analysis files available on this machine.

In [ ]:
exp_search = ra.get_datasets_from_protocol_names('spatialnoise')
available_experiments = os.listdir(ra.ANALYSIS_DIR)
exp_search = exp_search.query('exp_name in @available_experiments').reset_index(drop = True)
ra.scrollable_dataframe(exp_search)

## Plot Mosaics for all the Options

Plots RF mosaics for every classified noise chunk found in `exp_search`, split by cell type.


In [ ]:
cell_types = None  # None = auto-detect; or e.g. ['on/brisk sustained']
EXCLUDE_CELL_TYPES = ['unknown', 'nc', 'large', 'big', 'huge', 'weak']  # dropped from auto-detected labels

all_axes = ra.plot_mosaics_for_datasets(exp_search, cell_types=cell_types,
                                        exclude_cell_type_keywords=EXCLUDE_CELL_TYPES,
                                        minimum_n=3, b_zoom=True)


## Choose an Experiment

Pick which experiment to analyze and its SpatialNoise datafile (auto-picks the earliest if there are several).

In [ ]:
exp_name = '20260722A'
datafile_name = ra.find_datafile_for_protocol(exp_search, exp_name)

experiment_summary = ra.get_exp_summary(exp_name)
display(experiment_summary.head())


## Initialize Analysis Pipeline

Builds the MEA pipeline for this datafile, auto-picking a classified white-noise chunk for cell typing unless you set `MANUAL_ANALYSIS_CHUNK`.

In [ ]:
PREFERRED_TYPING_FILE = None  # e.g. 'data007.classificationYT.txt' -- None auto-picks the first one found
MANUAL_ANALYSIS_CHUNK = None  # e.g. 'data001' -- set to skip auto-detection

analysis_chunk_name = MANUAL_ANALYSIS_CHUNK or ra.find_classified_noise_chunk(exp_name)

with ra.scrollable_prints():
    pipeline = ra.create_mea_pipeline(
        exp_name, datafile_name, analysis_chunk_name=analysis_chunk_name, typing_file=PREFERRED_TYPING_FILE,
    )


## Break Out Underlying Objects

Splits the pipeline into its stimulus, response, and analysis-chunk objects for direct use below.

In [ ]:
stim_block = pipeline.stim
response_block = pipeline.resp
analysis_chunk = pipeline.analysis_chunk

## Show Stim Information

Looks at the raw per-epoch stimulus table and pulls out block-level parameters -- including the space-constant/stixel-size values, which this dataset stores as `stixelSizes` rather than `spaceConstants`.

In [ ]:
stimulus_data = stim_block.df_epochs
display(stimulus_data)

epoch_block_params = stim_block.d_epoch_block_params
all_space_constants = epoch_block_params['stixelSizes']
space_constant = stimulus_data['epoch_parameters'].apply(
    lambda x: x['stixelSize']
).values


Peek at the raw `epoch_parameters` dict for the first epoch, to see exactly what fields are available.

In [ ]:
stimulus_data['epoch_parameters'].iloc[0]


## Pull Spike Times and Spike Counts

Extracts per-cell spike times/counts for the chosen cell types, plus a baseline (pre-stimulus) spike count for comparison.

In [ ]:
spike_times = ra.get_spike_xarr(response_block, cell_types = cell_types, minimum_n = 3)
spike_counts = xr.apply_ufunc(len, spike_times, vectorize = True)

def get_avg_baseline(arr, pre_time):
    pre_spikes = [spike for spike in arr if spike < pre_time]
    return len(pre_spikes)

pre_time = epoch_block_params['preTime']
baseline_spike_counts = xr.apply_ufunc(get_avg_baseline, spike_times,
                                       kwargs = {'pre_time' : pre_time},
                                       vectorize = True)
